# Emisiones de una turbina de gas
## Carga de los CSV y variables

Empezamos como en los ejercicios del profesor: imports, lectura con `pd.read_csv` y un DataFrame llamado `df`.

Unimos los cinco archivos anuales apilando sus filas y conservando el año de origen. Todavía no limpiamos, graficamos, convertimos unidades, separamos X/y ni entrenamos modelos.

## 1. Importar las librerías

Las librerías habituales de clase. Las gráficas se trabajarán más adelante.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

## 2. Cargar los cinco CSV originales

Cada archivo corresponde a un año, de 2011 a 2015. Los cinco tienen las mismas once columnas y ya incluyen el encabezado en la primera línea.

La ruta permite ejecutar desde la raíz del proyecto o desde `notebooks/`. Si faltan archivos, ejecutar primero `uv run python -m scripts.descargar_datos` desde la raíz.

In [ ]:
ruta_datos = Path('data/raw')
if not ruta_datos.is_dir():
    ruta_datos = Path('../data/raw')

df_2011 = pd.read_csv(ruta_datos / 'gt_2011.csv')
df_2012 = pd.read_csv(ruta_datos / 'gt_2012.csv')
df_2013 = pd.read_csv(ruta_datos / 'gt_2013.csv')
df_2014 = pd.read_csv(ruta_datos / 'gt_2014.csv')
df_2015 = pd.read_csv(ruta_datos / 'gt_2015.csv')

## 3. Conservar el año de procedencia

Añadimos `anio` a cada tabla en memoria. No es un sensor ni una variable elegida para predecir: nos permite saber de qué archivo viene cada registro y respetar el tiempo en una evaluación futura.

No modificamos los CSV originales ni inventamos una fecha completa para cada fila.

In [ ]:
df_2011['anio'] = 2011
df_2012['anio'] = 2012
df_2013['anio'] = 2013
df_2014['anio'] = 2014
df_2015['anio'] = 2015

## 4. Armar un solo dataset: `df`

`pd.concat` apila las filas de los cinco años. `ignore_index=True` genera un índice consecutivo; no mezcla ni ordena aleatoriamente los registros.

Esperamos **36,733 filas y 12 columnas**: once variables originales y `anio`. No guardamos otro CSV; por ahora el dataset unido vive en memoria.

In [ ]:
df = pd.concat([df_2011, df_2012, df_2013, df_2014, df_2015],
               ignore_index=True)
df.shape

In [ ]:
df.head()

## 5. Lista de columnas

Los nombres se conservan como aparecen en los originales; sólo se añadió `anio`.

In [ ]:
variables = df.columns.tolist()

for numero, variable in enumerate(variables, start=1):
    print(f'{numero}. {variable}')

## 6. ¿Qué significa cada variable?

Las descripciones provienen de [UCI](https://archive.ics.uci.edu/dataset/551/gas+turbine+co+and+nox+emission+data+set) y del [artículo original](https://journals.tubitak.gov.tr/elektrik/vol27/iss6/53). Su figura 2 muestra sensores y parámetros del sistema.

| Columna | Significado | Unidad publicada / precaución |
|---|---|---|
| AT | Temperatura ambiente | °C |
| AP | Presión atmosférica | mbar |
| AH | Humedad ambiente | % |
| AFDP | Diferencia de presión en el filtro de aire | mbar; caída de presión a través del filtro |
| GTEP | Presión de escape de la turbina de gas | mbar; no asumir referencia absoluta/manométrica |
| TIT | Temperatura de entrada a la turbina | °C; no es directamente temperatura de llama |
| TAT | Temperatura después de la turbina | °C |
| TEY | Producción energética de la turbina | MWh en la fuente; no asumir potencia ni eficiencia |
| CDP | Presión de descarga del compresor | Unidad contradictoria: mbar en tabla, bar en esquema |
| CO | Concentración de monóxido de carbono en escape | mg/m³; no confundir con CO₂ |
| NOX | Concentración de óxidos de nitrógeno, descritos como NO + NO₂ | mg/m³ |
| anio | Año del archivo de origen | Procedencia añadida; no medición física |

**Sentido de ingeniería:** aire ambiente → filtro → compresor → combustión → turbina → escape, con generación eléctrica asociada. Estas variables describen condiciones ambientales, estado operativo y emisiones, no causas demostradas por sí mismas.

### Precauciones antes de analizar

- **CDP:** no cambiamos los valores ni convertimos unidades. La contradicción de la documentación queda pendiente de aclarar.
- **TEY:** conservamos la denominación de energía publicada. Interpretar potencia media requiere confirmar qué representa el agregado horario.
- **Tiempo:** no hay timestamp por fila. El orden es cronológico según la fuente, pero no inventamos una secuencia sin huecos.
- **Unidad de observación:** son registros horarios de una instalación, no 36,733 turbinas independientes.
- **Evaluación futura:** juntar los archivos no significa que debamos mezclar los años aleatoriamente. La referencia del artículo usa 2011–2012 para entrenamiento, 2013 para validación y 2014–2015 para prueba. Todavía no realizamos esa división.

**Hasta aquí llega este primer paso.** Una ruta posterior sería estimar NOX con variables ambientales y operativas, sin usar CO ni NOX como entradas. No hay aún selección de predictores, modelos o métricas.

Detalle documental: [diccionario de variables](../docs/VARIABLES.md).